In [ ]:
import os, sys
from os import listdir
from os.path import join, basename, dirname
from datetime import datetime, timezone, timedelta
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F
import torchaudio
import torchaudio.transforms as T
from torch.utils.data import Dataset, DataLoader
from torchvision.models import resnet18, ResNet18_Weights
from pytorch_metric_learning import losses
import numpy as np
import numpy.random as npr
from sklearn.model_selection import train_test_split

from shared import Data

device = torch.device('cuda')

def elapsed():
    return datetime.now(timezone(timedelta(hours=7))).strftime("%H:%M:%S")

# os.makedirs('/kaggle/working/cache', exist_ok=True)

In [ ]:
class Data2(Data):

    def __init__(
            self,
            sr: int = 32_000,
            is_train: bool = True
            ):
        super().__init__(sr, is_train)

        self._train_audio = r'birdclef-2026/data/train_soundscapes'
        self._train_labels = r'birdclef-2026/data/train_soundscapes_labels.csv'

        # hardcode, потому что требуется по 5сек
        duration = 5
        self._segment_len = int(duration * sr)
        # аналогично 60 / 5
        self._num_segments = 12

        # по записям
        with open(r'data/train_soundscapes_labels.csv', 'r', encoding='utf-8') as f:
            
            for line in f:

                # [BC2026_Test_<file ID>_<site>_<date>_<time in UTC>.ogg] [start] [end] [label1;label2;label3;...]
                audio, start, end, labels = line.split(',')

                # потому что мульти-классовое
                self._samples.append((
                    join(self._train_audio, audio),     # *Момент в том, что нет привязки ко времени
                    self._parsing_labels(labels)
                ))

        train_samples, valid_samples = train_test_split(
            self._samples,
            test_size=.2,
            random_state=42
        )
        self._samples = train_samples if self._is_train else valid_samples


    def _parsing_labels(self, labels: str):
        '''Создает multi-hot vector длины 234'''

        labels_ = labels.strip().split(';')
        vector = torch.zeros(234)

        for label in labels_:
            vector[self._cls2idx[label]] = 1
        return vector

    def _make_slides(self, y):
        
        segments = []
        for start in range(self._num_segments):
            segment = y[start:start+self._segment_len]
            segments.append(self._get_sps(segment))

        return torch.stack(segments)

In [ ]:
batch_size = 4
train_dataset = DataLoader(Data2(), batch_size=batch_size, num_workers=4, shuffle=True, pin_memory=True)
valid_dataset = DataLoader(Data2(is_train=False), batch_size=batch_size, num_workers=4, shuffle=False, pin_memory=True)

In [ ]:
model = resnet18()

old_conv = model.conv1
model.conv1 = nn.Conv2d(2, 64, 7, 2, 3, bias=False)

with torch.no_grad():
    model.conv1.weight[:] = old_conv.weight[:, :2]

model.fc = nn.Linear(512, 234)

In [ ]:
state = torch.load('best_label.pth', map_location=device)
model.load_state_dict(state['model'], strict=False)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=3e-4
)
criterion = nn.BCEWithLogitsLoss().to(device)
model.to(device);

In [ ]:
green = "\033[92m"
reset = "\033[0m"

EPOCH = 10
total_train, total_valid = len(train_dataset), len(valid_dataset)

best_loss = float('inf')

for epoch in range(EPOCH):

    # train
    train_loss = 0.0
    for xb,yb in train_dataset:

        B, N, C, H, W = xb.shape

        x = xb.view(B*N, C, H, W).to(device, non_blocking=True)
        y = yb.unsqueeze(1).repeat(1, N, 1).view(B*N, 234)
        y = y.to(device, non_blocking=True)

        optimizer.zero_grad()

        logits = model(x)

        loss = criterion(logits, y)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    train_loss /= total_train

    # valid
    valid_loss = 0.0
    with torch.no_grad():

        for xb, yb in valid_dataset:

            B, N, C, H, W = xb.shape

            x = xb.view(B*N, C, H, W).to(device, non_blocking=True)

            y = yb.unsqueeze(1).repeat(1, N, 1).view(B*N, 234)
            y = y.to(device, non_blocking=True)

            logits = model(x)

            loss = criterion(logits, y)

            valid_loss += loss.item()

    valid_loss /= total_valid

    print(f'Epoch: [ {epoch+1:^2} / {EPOCH} ]   [{elapsed()}]')
    print(f'  TrainLoss: {green}{train_loss:.4f}{reset},   ValidLoss: {valid_loss:.4f}')

    if valid_loss < best_loss:
        best_loss = valid_loss

        torch.save({
            'model': model.state_dict()
        }, 'best_multi.pth')
        
        print(f'  Модель {epoch+1} эпохи сохранена')